# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedr456/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The queue an editor actually receives: the model's out-of-fold score orders it, and every
row carries **one reason code and one action in plain words** — the reason-code taxonomy I
engineered in my Prompt Ladder exercise (Week 2), now running on real output. Codes are
built only from observable, safe facts (never from the label):

| code | fires when | action |
|---|---|---|
| `STALE_VISIBLE` | no update ≥180d AND ≥100 impressions/90d | REFRESH |
| `UNDER_CLICKING` | CTR < 50% of its position band's median | METADATA_REVIEW |
| `PAGE1_AT_RISK` | ranks ≤10 with a high model score | PROTECT_REVIEW |
| `LOW_ENGAGEMENT` | visible but engagement below panel median | CONTENT_REVIEW |
| `MODEL_FLAG` | high score, no single rule explains it | REVIEW |

Tiebreak (from the ladder's overlap check): the more specific code wins, in the table's
order. One code per row — a trustworthy queue explains itself in one line.

In [1]:
# Rebuild OOF scores (same seed/design as ML-08), attach reason codes, rank.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mohamedr456/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "matplotlib"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = (df["trend_direction"].str.lower() == "down").astype(int)
X = pd.DataFrame({
    "content_age_days": df["content_age_days"],
    "days_since_last_update": df["days_since_last_update"],
    "log_impressions_90d": np.log1p(df["impressions_90d"]),
    "avg_position": df["avg_position"].replace(0, np.nan),
    "has_position": (df["avg_position"] > 0).astype(int),
    "ctr": df["ctr"],
    "word_count": df["word_count"],
    "has_word_count": df["word_count"].notna().astype(int),
    "engagement_rate": df["engagement_rate"],
})
X["avg_position"] = X["avg_position"].fillna(X["avg_position"].median())
X["word_count"] = X["word_count"].fillna(X["word_count"].median())
X["engagement_rate"] = X["engagement_rate"].fillna(0)

oof = np.zeros(len(X))
imps = np.zeros(X.shape[1])
for tr, te in GroupKFold(5).split(X, y, groups=df["client_id"]):
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                n_jobs=-1, random_state=42).fit(X.iloc[tr], y.iloc[tr])
    oof[te] = rf.predict_proba(X.iloc[te])[:, 1]
    imps += rf.feature_importances_ / 5

# reason codes from observable facts only
pos_ok = df["avg_position"] > 0
band = pd.cut(df["avg_position"].where(pos_ok), [0, 3, 10, 20, 1000],
              labels=["top3", "4-10", "11-20", "21+"])
band_med = df.groupby(band, observed=True)["ctr"].transform("median")
hi_score = oof >= np.quantile(oof, 0.90)

code_ = np.select(
    [ (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 100),
      pos_ok & (df["ctr"] < 0.5 * band_med),
      pos_ok & (df["avg_position"] <= 10) & hi_score,
      (df["impressions_90d"] >= 100) & (df["engagement_rate"] < df["engagement_rate"].median()),
      hi_score ],
    ["STALE_VISIBLE", "UNDER_CLICKING", "PAGE1_AT_RISK", "LOW_ENGAGEMENT", "MODEL_FLAG"],
    default="NONE")
action = pd.Series(code_).map({"STALE_VISIBLE": "REFRESH", "UNDER_CLICKING": "METADATA_REVIEW",
                               "PAGE1_AT_RISK": "PROTECT_REVIEW", "LOW_ENGAGEMENT": "CONTENT_REVIEW",
                               "MODEL_FLAG": "REVIEW", "NONE": "MONITOR"})

queue = (df[["content_id", "client_id", "content_type", "days_since_last_update",
             "impressions_90d", "avg_position", "ctr", "engagement_rate"]]
         .assign(model_score=np.round(oof, 4), reason_code=code_, action=action.values)
         .sort_values("model_score", ascending=False).reset_index(drop=True))
print(queue.head(15).to_string(index=False))
print("\nreason codes in the top 100:")
print(queue.head(100)["reason_code"].value_counts().to_string())

          content_id         client_id    content_type  days_since_last_update  impressions_90d  avg_position  ctr  engagement_rate  model_score    reason_code          action
content_78a8704f334f client_349c41201b keyword article                      20             3818          20.0 0.10             2.13       0.9784     MODEL_FLAG          REVIEW
content_37f4883d6bbe client_349c41201b keyword article                      20             4422          16.0 0.11             0.00       0.9745     MODEL_FLAG          REVIEW
content_a8cd736c9c7b client_349c41201b keyword article                      20             2877          23.2 0.03             0.00       0.9709     MODEL_FLAG          REVIEW
content_f2b9744f016d client_349c41201b keyword article                      20             5503          28.8 0.07             0.00       0.9700     MODEL_FLAG          REVIEW
content_9bf53d7bbea5 client_4ec9599fc2  feedly article                      20              334          29.6 0.00      

## 2. Intended use and limits

**Who:** a FlyRank content strategist planning the week's refresh work across a client
portfolio. **What:** take the top of the queue, read each row's reason code, and decide —
the score orders the work, the human owns the call. Capacity-fit: the top 50 is one
editor-week; precision there is 0.70 under client-holdout vs 0.542 base.

**Where it stops being valid:**
- It ranks **observed, concurrent decline** in a June-2026 snapshot — it does not forecast
  next quarter, and it does not survive unchanged into a different season or portfolio mix.
- Scores transfer to *unseen clients* at the 0.70 level on average, with wide client-mix
  variance (fold spread 0.56–0.78) — small or unusual clients get noisier queues.
- No page content, no queries in the clear: the model knows reach/position/age, never
  *why* — so the reason code is a starting hypothesis, not a diagnosis.

In [2]:
# The capacity-fit numbers the intended-use section quotes, recomputed live.
def p_at_k(s, l, k):
    o = np.argsort(-np.asarray(s)); return float(np.asarray(l)[o[:k]].mean())
print("queue precision (out-of-fold, whole panel pooled):")
for k in (10, 20, 50, 100):
    print(f"  top-{k:<4} observed-declining share: {p_at_k(oof, y, k):.3f}")
print(f"  base rate: {y.mean():.3f}")

queue precision (out-of-fold, whole panel pooled):
  top-10   observed-declining share: 0.700
  top-20   observed-declining share: 0.700
  top-50   observed-declining share: 0.700
  top-100  observed-declining share: 0.690
  base rate: 0.542


## 3. Human review + the no-go list

**Before acting on any row, a human checks:** (1) open the page — is it genuinely stale or
an evergreen reference? (2) does the reason code match what you see (an `UNDER_CLICKING`
page whose snippet already looks right means the model is guessing)? (3) is the client
context normal this window (migrations, seasonality, tracking changes)?

**Never automated, ever:**
- **No auto-rewrites or auto-publishing.** The queue orders human attention; it never
  edits content.
- **No deletions or prunes from a score.** A low score is absence of signal, not a verdict.
- **No client-facing claims from reason codes** ("your page is failing because…") — codes
  are internal hypotheses in careful language.
- **No cross-client comparisons** ("client A's content beats client B's") — pseudonymized
  panels with different history depths cannot carry that sentence.

In [3]:
# Guard in code: the no-go list as assertions on the export itself.
assert "trend_direction" not in queue.columns and "trend_pct" not in queue.columns
assert queue["model_score"].between(0, 1).all()
print("export carries no label columns and no client-identifying fields:")
print(list(queue.columns))

export carries no label columns and no client-identifying fields:
['content_id', 'client_id', 'content_type', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'model_score', 'reason_code', 'action']


## 4. Monitoring / retrain triggers

The queue goes stale silently; these are the tripwires, each checkable from the same data
the pipeline already reads:

1. **Precision decay:** monthly, re-score and measure top-50 observed-decline share; two
   consecutive months below the rule baseline (0.56) → retrain.
2. **Score drift:** population stability of the score distribution (PSI > 0.2 vs the
   training month) → investigate before trusting ranks.
3. **Mix shift:** share of `has_position = 0` rows or content-type mix moves ±10 pts →
   features mean something different now; revalidate.
4. **Reason-code collapse:** if one code exceeds ~70% of the top 100 (today: mixed), the
   taxonomy stopped explaining the queue → revisit thresholds.
5. **Panel events:** any client onboarding/tracking change (new `ga4_data_start`) →
   exclude that client's rows from evaluation for one window.

In [4]:
# Trigger 4 checked today, as the example of a live tripwire.
top100_share = queue.head(100)["reason_code"].value_counts(normalize=True)
print("top-100 reason-code shares today:")
print(top100_share.round(2).to_string())
print("\nmax share:", round(float(top100_share.max()), 2), "-> tripwire (0.70) not hit today")

top-100 reason-code shares today:
reason_code
MODEL_FLAG        0.54
UNDER_CLICKING    0.34
PAGE1_AT_RISK     0.12

max share: 0.54 -> tripwire (0.70) not hit today


## 5. Exports for the paper

Written below: the full ranked queue CSV (regenerated every run, out of git by the CI
leak-guard), the playbook metrics JSON (committed — receipts), and the three figures the
deployed paper embeds (committed under `work/outputs/figs/`).

In [5]:
# Queue CSV + metrics JSON + the paper's three figures.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import json as _json

os.makedirs("work/outputs/figs", exist_ok=True)
queue.to_csv("work/outputs/action_queue.csv", index=False)

metrics = {"top_k_precision": {str(k): round(p_at_k(oof, y, k), 3) for k in (10, 20, 50, 100)},
           "base_rate": round(float(y.mean()), 3),
           "reason_code_top100": queue.head(100)["reason_code"].value_counts().to_dict(),
           "seed": 42}
with open("work/outputs/playbook_metrics.json", "w") as f:
    _json.dump(metrics, f, indent=2)

# Fig 1 — model vs baselines at P@50 (numbers from ML-08 receipts)
m = _json.load(open("work/outputs/model_metrics.json"))["means"]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
names = ["random order", "frozen rule", "logistic reg.", "random forest"]
vals = [m["p50_random"], m["p50_rule"], m["p50_logreg"], m["p50_rf"]]
bars = ax.bar(names, vals, color=["#9ca3af", "#9ca3af", "#9ca3af", "#0F766E"])
ax.axhline(m["base_rate"], ls="--", c="#111827", lw=1, label=f"base rate {m['base_rate']}")
ax.bar_label(bars, fmt="%.2f"); ax.set_ylim(0, 1)
ax.set_ylabel("Precision@50 (client-holdout)"); ax.legend(frameon=False)
ax.set_title("Who fills a 50-page refresh queue best?")
plt.tight_layout(); plt.savefig("work/outputs/figs/fig1_p50_comparison.png", dpi=150); plt.close()

# Fig 2 — observed decline rate by model-score decile (lift shape)
dec = pd.qcut(oof, 10, labels=False, duplicates="drop")
lift = y.groupby(dec).mean()
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(lift.index.astype(int) + 1, lift.values, color="#0F766E")
ax.axhline(y.mean(), ls="--", c="#111827", lw=1, label=f"base rate {y.mean():.2f}")
ax.set_xlabel("model-score decile (10 = highest score)")
ax.set_ylabel("observed decline rate"); ax.legend(frameon=False)
ax.set_title("Decline rate climbs with the model score (out-of-fold)")
plt.tight_layout(); plt.savefig("work/outputs/figs/fig2_lift_by_decile.png", dpi=150); plt.close()

# Fig 3 — what the model leans on
imp_s = pd.Series(imps, index=X.columns).sort_values()
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.barh(imp_s.index, imp_s.values, color="#0F766E")
ax.set_xlabel("mean feature importance (5 client-holdout folds)")
ax.set_title("Reach, position and age carry the model — staleness barely does")
plt.tight_layout(); plt.savefig("work/outputs/figs/fig3_importances.png", dpi=150); plt.close()

print("written: work/outputs/action_queue.csv (regenerated, not committed)")
print("committed receipts + figures:")
for p in ["work/outputs/playbook_metrics.json", "work/outputs/figs/fig1_p50_comparison.png",
          "work/outputs/figs/fig2_lift_by_decile.png", "work/outputs/figs/fig3_importances.png"]:
    print("  ", p, os.path.getsize(p), "bytes")

written: work/outputs/action_queue.csv (regenerated, not committed)
committed receipts + figures:
   work/outputs/playbook_metrics.json 230 bytes
   work/outputs/figs/fig1_p50_comparison.png 36027 bytes
   work/outputs/figs/fig2_lift_by_decile.png 34300 bytes
   work/outputs/figs/fig3_importances.png 42362 bytes


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.